# 02 Entity Recognition and Further Filtering

For assignment, to answer the impact of the AI, we need to identify the following aspects:
- **Affected Entity**: The name of the organization or company that uses or affected by AI;
- **Industry**: The industry this article is related to;
- **Technology**: The specific AI technology mentioned in the article;

Futhermore, to help with or cross validate sentimental analysis (impact of AI) in later steps, we can also let LLM to identify the sentiment of the article, which can be categorized into "positive", "neutral" and "negative".

In [ ]:
import json
import os
import polars as pl
import zipfile

from embedding import get_embedder
from gemini_utility import GeminiInstructJsonClient
from pydantic import BaseModel
from typing import Literal

## 2.1 NER Using Gemini

In [ ]:
class KeyExtraction(BaseModel):
    organization: str
    industry: str
    impact: Literal[
        'strong positive',
        'positive',
        'neutral',
        'negative',
        'strong negative'
    ]
    technology: str

In [ ]:
instruct_model = 'gemini-2.5-flash-lite'
instruction = '''
Extract the key points from the following article summary related to AI:
- organization: The name of the organization or company that uses or affected by AI;
- industry: The industry this article is related to;
- impact: The impact of AI is: strong positive, positive, neutral, negative, strong negative;
- technology: The specific AI technology mentioned in the article; if not specific, return "AI".;
'''.strip() + "\n\n"

resp_cache_dir = 'ner'

article_df = pl.read_parquet('011_summary.parquet')
print(f'Number of articles: {article_df.height}')
texts = article_df.get_column("summary").to_list()

In [ ]:
# Uncomment the whole cell to execute the task and save the cache.

# task_client = GeminiInstructJsonClient(
#     llm_model=instruct_model,
#     instruction=instruction,
#     resp_model=KeyExtraction
# )

# task_client.execute_and_save_cache(
#     cache_dir=resp_cache_dir,
#     texts=texts,
#     batch_size=1000,
#     max_workers=48
# )

# # Store cache

# with zipfile.ZipFile('ner.zip', 'w') as zipf:
#     for root, dirs, files in os.walk(resp_cache_dir):
#         for file in files:
#             zipf.write(os.path.join(root, file), arcname=file)

In [ ]:
# Load from cache

with zipfile.ZipFile('ner.zip', 'r') as zipf:
    zipf.extractall(resp_cache_dir)

cache_files = sorted(list(os.listdir(resp_cache_dir)))
extraction_responses = []
error_count = 0

for file in cache_files:
    with open(os.path.join(resp_cache_dir, file), 'r') as f:
        for line in f:
            res = json.loads(line)
            if res["error"]:
                error_count += 1
            extraction_responses.append(res)

print(f"Total responses: {len(extraction_responses)}, Errors: {error_count}")

# Total responses: 164091, Errors: 1
# The only error here is because of model moderation. See `texts[49557]`.

## 2.2 Merge dataset

In [ ]:
dates = article_df.get_column("date").to_list()
titles = article_df.get_column("title").to_list()

data = []
for idx, res in enumerate(extraction_responses):
    if res["error"]:
        continue
    data.append({
        "date": dates[idx],
        "title": titles[idx],
        "summary": texts[idx],
        **res["res"]
    })

df = pl.DataFrame(data)
impact_map = {
    'strong positive': 2,
    'positive': 1,
    'neutral': 0,
    'negative': -1,
    'strong negative': -2
}

df = df.with_columns(
    pl.col("impact").replace_strict(impact_map, return_dtype=pl.Int8).alias("impact")
)
df.write_parquet('020_labeled_data.parquet')
print('Number of records:', df.height)

In [ ]:
df.head()

## 2.3 Further Filtering and Cleaning

In [ ]:
# Filter out articles with "N/A" in industry or technology
# which are likely not relevant to our analysis.

df = pl.read_parquet('020_labeled_data.parquet')

filtered_df = df.filter(
    (pl.col("industry") != "N/A") &
    (pl.col("technology") != "N/A")
)

# Remove '/Technology' suffix or 'Technology/' prefix
filtered_df = filtered_df.with_columns(
    pl.col("industry")
        .str.replace(r'\b[Tt]ech\b', 'Technology')
        .str.replace(' / ', '/')
        .str.replace(r'/[Tt]echnology', '')
        .str.replace(r'[Tt]echnology/', '')
        .str.replace(r'[Tt]echnology, ', '')
        .str.replace(r', [Tt]echnology', '')
        .str.replace(r'[Tt]echnology and ', '')
        .str.strip_chars()
        .alias("industry")
)

print('Number of records:', filtered_df.height)
filtered_df.write_parquet('021_filtered_data.parquet')